<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />

# Worksheet 11.2: Building & Using an Agent Tool

A large language model can only *talk* until you give it **tools** — plain
Python functions it can call to actually *do* things. In this lab you'll take a
function that scores how suspicious a domain looks and turn it into a tool that
Claude can call on its own.

The security scoring is given to you. The point of this lab is the **mechanics
of tool use**: writing the docstring the model reads, binding the tool to an
agent, invoking it, and watching the agent decide to call it.


In [ ]:
# Load Libraries - Make sure to run this cell!
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent

# Loads ANTHROPIC_API_KEY from your .env file in the project root.
load_dotenv()

MODEL = "claude-opus-4-8"


## The function you'll wrap

First, a plain function. Nothing AI about it yet — it takes a domain string and
returns a verdict string. Run it and confirm it works.

In [ ]:
# GIVEN — you do NOT need to understand this deeply.
# A few cheap, explainable heuristics that flag algorithmically-generated
# ("DGA") or otherwise sketchy domains. This is the same idea as Worksheet 4.1.
from collections import Counter
import math

def _shannon_entropy(s: str) -> float:
    """Randomness of a string. DGA domains score high (~3.5+)."""
    if not s:
        return 0.0
    return -sum((n / len(s)) * math.log2(n / len(s)) for n in Counter(s).values())

def score_domain(domain: str) -> str:
    """Return a plain-string verdict + reasons for a domain."""
    domain = domain.strip().lower()
    for prefix in ("https://", "http://"):
        if domain.startswith(prefix):
            domain = domain[len(prefix):]
    domain = domain.split("/")[0]

    label = domain.split(".")[0]
    reasons = []
    entropy = _shannon_entropy(label)
    if entropy > 3.5:
        reasons.append(f"high entropy ({entropy:.2f}) — possible DGA")
    if len(domain) > 30:
        reasons.append(f"unusually long ({len(domain)} chars)")
    digits = sum(ch.isdigit() for ch in domain)
    if digits and digits / len(domain) > 0.3:
        reasons.append(f"digit-heavy ({digits} digits)")
    if domain.rsplit(".", 1)[-1] in {"zip", "mov", "xyz", "top", "tk"}:
        reasons.append("high-abuse TLD")

    verdict = "SUSPICIOUS" if reasons else "likely benign"
    detail = "; ".join(reasons) if reasons else "no red flags"
    return f"{domain}: {verdict}. {detail}"

# Sanity check — it's just a function that takes a string and returns a string.
print(score_domain("google.com"))
print(score_domain("kq3v9z7bnx4p2wlm8.top"))


## Step 1 — Turn the function into a tool

The `@tool` decorator (from LangChain) turns a normal function into something an
agent can call. The function's **docstring becomes the tool's description** —
this is the text Claude reads to decide *whether* and *how* to use it. A vague
docstring means the agent won't know when to call your tool.

In [ ]:
# Turn score_domain into something Claude can call.
# The DOCSTRING is not a comment — it is the tool's API. The agent reads it to
# decide WHEN to call the tool and WHAT to pass. Write it for the model.

# TODO: add the decorator that turns this function into an agent tool.
____
def check_domain_reputation(domain: str) -> str:
    # TODO: write a docstring that tells the agent WHEN to use this tool and
    # WHAT argument to pass. This is the most important line in the lab.
    """____"""
    return score_domain(domain)

# Self-check: @tool wraps the function and exposes its name + description.
assert check_domain_reputation.name == "check_domain_reputation"
assert check_domain_reputation.description  # the docstring the agent reads
print("Tool name:", check_domain_reputation.name)
print("What the agent sees:\n", check_domain_reputation.description)


## Step 2 — Build an agent that has the tool

`create_agent` wires your Claude model together with a list of tools. The
agent will reason about each question and call a tool only when it helps.

In [ ]:
# Give the model your tool. create_agent lets Claude decide, on its own,
# whether a question needs the tool.
llm = ChatAnthropic(model=MODEL, api_key=os.getenv("ANTHROPIC_API_KEY"))

# TODO: build the agent. Pass your tool to create_agent inside the list.
agent = create_agent(llm, [____])
print("Agent ready with tools:", [check_domain_reputation.name])


## Step 3 — Use it

Ask a question. You pass a *user message*, not a function call — the agent looks
at your tool's description and decides for itself to call it.

In [ ]:
# Ask the agent about a sketchy-looking domain. Notice you never call the tool
# yourself — the agent reads your docstring and decides to call it.

# TODO: invoke the agent with a user question about a suspicious domain,
# then print the final answer (the last message's .content).
result = agent.invoke(
    {"messages": [("user", "____")]}
)
print(result["messages"][-1].____)


## Step 4 — See the tool call

The final answer hides the interesting part. Walk the message history to see
that the agent actually *chose* to call your tool, what argument it passed
(pulled from the docstring you wrote), and what came back.

In [ ]:
# GIVEN — walk the message history so you can SEE that the agent chose to
# call your tool, what it passed in, and what came back.
for msg in result["messages"]:
    role = msg.__class__.__name__
    if role == "AIMessage" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[agent decided to call] {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"[tool returned]           {msg.content}")
    elif role == "AIMessage" and msg.content:
        print(f"[final answer]            {msg.content}")


## From tool to skill

You just built a **tool**: a single capability the agent can call. A **skill**
is the layer above it — the *analyst judgment and house format* wrapped around
one or more tools.

Look at `../ai_examples/skills/triage-suspicious-url/SKILL.md`. It doesn't add
any new capability. It tells the agent **when** to reach for a reputation
check, to **defang** indicators before echoing them (`evil.com` → `evil[.]com`),
and what **verdict format** to emit. Same tool, wrapped in a repeatable workflow.

- **Tool** = *what the agent can do* (your `check_domain_reputation`).
- **Skill** = *when and how it should do it* (the SOC triage playbook).


In [ ]:
# Read the paired skill that would wrap this tool in a real SOC workflow.
from pathlib import Path
print(Path("../ai_examples/skills/triage-suspicious-url/SKILL.md").read_text())
